# CONFIGURAÇÃO FINAL — Pair-Aware Linear sem pesos, P(neg) ≥ 0,80 — RECLin-PT (Colab T4)

**Prepara o congelamento da configuração final. O TEST não é lido nem avaliado neste notebook.**

A configuração final (emenda de 22/11/2026, `codigo/docs/protocolo/emenda-configuracao-final.md`) é:

* o modelo da execução `pair_aware__a3ac671a77e2__t2` (Pair-Aware Linear, BioBERTpt-all, semente 42, **sem pesos de classe**, 3 épocas, melhor época = 3), **sem treino novo**;
* a regra de decisão: `negation_of` quando P(neg) ≥ t; senão, o argmax entre `associated_with` e `no_relation`;
* t escolhido no DEV pelo critério declarado: **maior precisão de `negation_of` entre os limiares da grade 0,05…0,95 com recall ≥ 0,80** (esperado: t = 0,80; P 0,845, R 0,800, F1 0,822).

O que este notebook faz, em ordem:

1. confere os pesos da execução no Drive contra os SHA-256 gravados no fim do treino;
2. **reempacota** a execução sob o código atual (o treino foi feito com o código `0526d912…`, anterior à trava do TEST): recarrega os pesos, prevê o DEV e exige as mesmas classes em todos os 19.064 pares (diferença de probabilidade ≤ 0,001). Só então cria `pair_aware_sem_pesos_biobertpt_seed42`, com uma cópia conferida dos pesos;
3. escolhe e congela t no DEV (`pair_aware_limiar_sem_pesos_biobertpt_seed42/limiar.json`);
4. grava o registro de congelamento (`configuracao_final.json`) e mostra o código de liberação;
5. (opcional, com confirmação) envia os resultados do DEV à branch `resultados-colab`.

**Antes de rodar:** o código desta etapa precisa estar no `main` do GitHub (commit e push feitos por você).

**Depois:** a avaliação única do TEST só acontece com a sua autorização explícita (seção 9).

## 1. Clonar o repositório

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

try:
    userdata.get('GITHUB_PAT')               # secret do Colab, nunca hard-coded (o valor não é guardado nem impresso)
except Exception as erro:
    raise SystemExit('Secret GITHUB_PAT indisponível (%s). Colab: ícone de chave → "+ Add new secret" → nome '
                     'GITHUB_PAT → cole o token → ligue "Notebook access" → rode esta célula de novo.' % type(erro).__name__)
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

# O token NÃO entra na URL (ficaria gravado em .git/config): vai por GIT_ASKPASS (script 700, apagado ao
# terminar), só no ambiente do processo do git.
url = f'https://github.com/angeloalsf/{REPO_NAME}.git'

def git(*args, check=True):
    pasta = tempfile.mkdtemp(prefix='askpass-'); os.chmod(pasta, 0o700)
    script = os.path.join(pasta, 'askpass.sh')
    with open(script, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in *sername*) printf "x-access-token";; *) printf "%s" "$RECLIN_GIT_TOKEN";; esac\n')
    os.chmod(script, 0o700)
    env = {**os.environ, 'GIT_ASKPASS': script, 'GIT_TERMINAL_PROMPT': '0', 'RECLIN_GIT_TOKEN': userdata.get('GITHUB_PAT')}
    try:
        return subprocess.run(['git', '-c', 'credential.helper=', *args], env=env, check=check)
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

if not os.path.isdir(REPO):
    git('clone', url, REPO)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    git('-C', REPO, 'pull', '--rebase', '--autostash', check=False)

git('-C', REPO, 'config', 'user.name', GIT_USER_NAME)
git('-C', REPO, 'config', 'user.email', GIT_USER_EMAIL)
git('-C', REPO, 'remote', 'set-url', 'origin', url)

# Código novo do projeto: fica em REPO/codigo. Registra o commit clonado (HEAD) na proveniência de cada execução.
BRANCH = subprocess.run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO}/codigo')
from reclin.experimento import apresentacao, codigo_colab, operacoes, plano, raiz, segredos
INFO = codigo_colab.clonar_codigo(f'angeloalsf/{REPO_NAME}', BRANCH, REPO)
SHA_CODIGO = INFO['sha']
CODIGO = Path(INFO['codigo'])
raiz.ativar(CODIGO)                          # põe codigo/ no sys.path (devolve o ambiente de subprocessos, não o caminho)
print('Repo pronto em', REPO, f'| branch {BRANCH} | commit {SHA_CODIGO}')

## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Montar o Google Drive

A execução de base fica onde o treino a deixou (`MyDrive/RECLin-PT/execucoes/pair_aware__a3ac671a77e2__t2`) e **não é alterada**. Tudo o que este notebook cria vai para `MyDrive/RECLin-PT/configuracao_final/` (as duas execuções novas, lado a lado, e o registro de congelamento).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ_DRIVE = Path(f'/content/drive/MyDrive/{REPO_NAME}')
BASE = RAIZ_DRIVE / 'execucoes' / 'pair_aware__a3ac671a77e2__t2'
RAIZ_FINAL = RAIZ_DRIVE / 'configuracao_final' / 'execucoes'
DESTINO_CONGELAMENTO = RAIZ_DRIVE / 'configuracao_final' / 'congelamento' / 'configuracao_final.json'
NOME_REEMPACOTADA = 'pair_aware_sem_pesos_biobertpt_seed42'
PARTICOES = Path(REPO) / 'codigo' / 'dados' / 'particoes-v2'     # só o DEV é lido
RAIZ_FINAL.mkdir(parents=True, exist_ok=True)
assert BASE.is_dir(), f'Base não encontrada no Drive: {BASE}'
print('Base:', BASE)
print('Saída:', RAIZ_FINAL)

## 4. Instalar dependências

In [ ]:
!pip install -q -r $REPO/codigo/requirements.txt

## 5. Conferir os arquivos da base (SHA-256 gravados no fim do treino)

Os valores abaixo são os do manifesto de sincronização do Drive (`SYNC.json` da execução, 08/11/2026) e os registrados em `experimento.json`. Qualquer divergência interrompe o notebook: os pesos não seriam os do treino analisado.

In [ ]:
import hashlib, json

def sha256(caminho, bloco=1 << 20):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for parte in iter(lambda: f.read(bloco), b''):
            h.update(parte)
    return h.hexdigest()

ESPERADOS = {
    'checkpoints/melhor_modelo/model.safetensors': '120e41f523b308979db1edecc9079c0ec789ff107ba24466f2a47e3f66257e76',
    'checkpoints/melhor_modelo/pair_aware_head.safetensors': '8eec71c603866e428c2cd458151939c37f2ccf0261a59f4bd6a5b0eb4562797b',
    'checkpoints/melhor_modelo/config.json': 'e61e256e5e6deeeb90fe80b96a1dc69430310f0e91575b823a4097b8ecfef6d5',
    'checkpoints/melhor_modelo/pair_aware_config.json': '10b48e6327dcb404cf005b0ed01cd151846ab989297eb2faae2394f67f286b14',
    'checkpoints/melhor_modelo/tokenizer.json': '9f1b6eeddc28ecafad39d5a47dba01138c6a2d8798be5815a1b9521895f9ae6a',
    'checkpoints/melhor_modelo/tokenizer_config.json': 'f401ca2dde9d387adfc1f4fa89a2355b63f37cadf55e7dbac7c3c308425b8767',
    'config.json': '19b1a9a351a55a4a142ae9cefc4c87250a33deb4fd2a9528b3819082d62fb760',
    'predicoes_dev.json': 'cf0fbda088cfa4f81806ea3f2ec25eb166ea99a8f294fb121342a03eae8ba558',
    'metricas.json': 'a846b853b896dffba2cfed2d54a5c5c7b7680bc6a08ac62ce9322771ac580665',
}
erros = []
for relativo, esperado in ESPERADOS.items():
    arquivo = BASE / relativo
    atual = sha256(arquivo) if arquivo.is_file() else 'AUSENTE'
    print(f"{'OK ' if atual == esperado else 'ERRO'} {relativo:<55} {atual[:16]}…")
    if atual != esperado:
        erros.append(relativo)
assert not (BASE / 'predicoes_test.json').exists(), 'O TEST da base já foi avaliado: pare aqui.'
_cfg = json.load(open(BASE / 'config.json', encoding='utf-8'))
assert _cfg['config']['config']['class_weight'] == 'none', 'A base não é a execução sem pesos de classe.'
assert not erros, f'Arquivos divergentes: {erros}'
print('\nBase conferida: execução sem pesos de classe, TEST nunca avaliado, pesos íntegros.')

## 6. Reempacotar sob o código atual (só DEV)

Recarrega o `melhor_modelo/` da base com o código atual, prevê o DEV inteiro e compara com o `predicoes_dev.json` da base. **Só se forem equivalentes** (mesma classe nos 19.064 pares; probabilidades com diferença ≤ 0,001) a execução nova é criada. Com qualquer divergência, nada é gravado e o notebook para: nesse caso, não siga adiante e me envie a mensagem.

In [ ]:
from reclin.estrategias import pair_aware
from reclin.treino import reempacotamento, reprodutibilidade
from reclin.util.io import ler_json

reprodutibilidade.configurar_ambiente()
REEMPACOTADA = RAIZ_FINAL / NOME_REEMPACOTADA
if (REEMPACOTADA / 'config.json').is_file():
    print('Já reempacotada em uma sessão anterior:', REEMPACOTADA)
    COMPARACAO = ler_json(REEMPACOTADA / 'treino.json')['reempacotamento']['comparacao_dev']
else:
    _montagem = pair_aware.montagem_de_registro(ler_json(BASE / 'config.json')['config'])
    _r = reempacotamento.reempacotar(BASE, RAIZ_FINAL, NOME_REEMPACOTADA, pasta_particoes=PARTICOES,
                                     montagem=_montagem)
    COMPARACAO = _r['comparacao']
print(json.dumps(COMPARACAO, indent=1, ensure_ascii=False))
assert COMPARACAO['equivalente'] and COMPARACAO['classes_divergentes'] == 0

## 7. Escolher e congelar t no DEV

Grade 0,05…0,95 (passo 0,05). Critério: **maior precisão de `negation_of` entre os limiares com recall de `negation_of` ≥ 0,80**; desempates: maior F1 de `negation_of`, maior macro-F1, menor limiar. O resultado esperado (curva de 17/11/2026) é t = 0,80 com 120 VP, 22 FP e 30 FN; a célula confere isso.

In [ ]:
from reclin.estrategias import pair_aware_limiar

NOME_FINAL = pair_aware_limiar.nome_execucao(NOME_REEMPACOTADA)
FINAL = RAIZ_FINAL / NOME_FINAL
if (FINAL / 'limiar.json').is_file():
    print('t já congelado em uma sessão anterior.')
    _lim = ler_json(FINAL / 'limiar.json'); T, TABELA = _lim['limiar'], _lim['tabela']
else:
    _r = pair_aware_limiar.calibrar_dev(REEMPACOTADA, RAIZ_FINAL, NOME_FINAL)
    T, TABELA = _r['t'], _r['tabela']

print(f"{'t':>5} {'VP':>4} {'FP':>4} {'FN':>4} {'P':>6} {'R':>6} {'F1':>6}  macro-F1")
for linha in TABELA:
    c = linha['contagens']['negation_of']
    p = c['tp'] / (c['tp'] + c['fp']) if c['tp'] + c['fp'] else 0.0
    r = c['tp'] / (c['tp'] + c['fn'])
    marca = '  <- escolhido' if linha['limiar'] == T else ''
    print(f"{linha['limiar']:5.2f} {c['tp']:4d} {c['fp']:4d} {c['fn']:4d} {p:6.3f} {r:6.3f} {linha['f1_negation_of']:6.3f}  {linha['macro_f1']:.4f}{marca}")
_c = next(l for l in TABELA if l['limiar'] == T)['contagens']['negation_of']
assert (T, _c['tp'], _c['fp'], _c['fn']) == (0.8, 120, 22, 30), 'O DEV não reproduz o ponto analisado: pare e me envie a saída.'
print('\nt = 0,80 confirmado no DEV.')

## 8. Congelar a configuração final

Grava o registro com as impressões digitais da configuração (limiar, predições do DEV, configuração e identidade do código) **e da base** (incluindo o hash dos pesos). O código de liberação aparece abaixo, mas **só deve ser usado depois da sua autorização explícita**, uma única vez.

In [ ]:
from reclin.execucao import congelamento as cg

RESPONSAVEL = 'Angelo Antonio Lima Silveira Filho'
JUSTIFICATIVA = ('Emenda de 22/11/2026: Pair-Aware Linear (BioBERTpt-all, semente 42) treinada sem pesos de classe, '
                 'com a regra P(negation_of) >= t; t escolhido no DEV, na grade 0,05-0,95, pela maior precisão de '
                 'negation_of com recall >= 0,80 (t = 0,80: P 0,845, R 0,800, F1 0,822 no DEV). Prioridade do projeto: '
                 'precisão de negation_of. Escolha feita só com o DEV.')
if DESTINO_CONGELAMENTO.is_file():
    print('Já existe um congelamento:', DESTINO_CONGELAMENTO, '(não é substituído).')
    REGISTRO = ler_json(DESTINO_CONGELAMENTO)
else:
    REGISTRO = cg.criar(FINAL, responsavel=RESPONSAVEL, justificativa=JUSTIFICATIVA, pasta_particoes=PARTICOES)
    cg.gravar(REGISTRO, DESTINO_CONGELAMENTO)
    REGISTRO = ler_json(DESTINO_CONGELAMENTO)
assert cg.impressoes_da_execucao(FINAL) == REGISTRO['execucao'], 'A configuração mudou depois do congelamento.'
print('Configuração congelada:', REGISTRO['execucao']['pasta'], '| base:', REGISTRO['execucao']['base']['pasta'])
print('Pesos da base (hash da pasta):', REGISTRO['execucao']['base']['sha256_melhor_modelo'][:16], '…')
print('Registro:', DESTINO_CONGELAMENTO)
print('Código de liberação:', REGISTRO['liberacao']['codigo'], '(use só depois da autorização do autor)')

## 9. Salvar os resultados do DEV no GitHub (opcional, com confirmação)

Envia à branch `resultados-colab` **só arquivos do DEV** das duas execuções novas: da reempacotada, `config.json`, `treino.json`, `predicoes_dev.json` e `metricas.json`; da configuração final, `config.json`, `limiar.json`, `predicoes_dev.json` e `metricas.json`. Pesos nunca vão. O registro de congelamento (`configuracao_final.json`) fica no Drive: baixe-o e coloque-o em `codigo/resultados/congelamento/` no seu repositório, quando decidir versioná-lo.

In [ ]:
from reclin.experimento import versionamento

PUBLICAR_NO_GITHUB = False          # True = confirmo enviar estes arquivos do DEV ao GitHub
BRANCH_RESULTADOS = 'resultados-colab'
ENVIOS = [(REEMPACOTADA, 'reempacotada'), (FINAL, 'derivada')]
for pasta, perfil in ENVIOS:
    versionamento.conferir(pasta, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS,
                           sha_codigo=SHA_CODIGO, perfil=perfil)
if not PUBLICAR_NO_GITHUB:
    print('\nNada foi enviado. Para enviar, ponha PUBLICAR_NO_GITHUB = True e rode de novo.')
else:
    _cred = segredos.Credencial('GITHUB_PAT', userdata.get('GITHUB_PAT'), 'Colab Secrets')
    try:
        for pasta, perfil in ENVIOS:
            versionamento.salvar(pasta, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS,
                                 sha_codigo=SHA_CODIGO, clone='/content/reclin_versionamento', nome_git=GIT_USER_NAME,
                                 email_git=GIT_USER_EMAIL, credencial=_cred, perfil=perfil)
        print('Push concluído (se houve mudanças).')
    except versionamento.ErroVersionamento as erro:
        print('ERRO:', segredos.limpar(str(erro), _cred))
    finally:
        _cred = None

## 10. Avaliação única do TEST — NÃO execute sem autorização explícita

Este notebook para aqui. Quando (e só quando) a avaliação final for autorizada, ela tem dois passos, sob a mesma liberação e uma única vez cada (a trava recusa qualquer outra ordem, outra execução ou uma segunda vez):

```
python codigo/scripts/avaliar_test.py --execucao <RAIZ_FINAL>/pair_aware_sem_pesos_biobertpt_seed42 \
    --congelamento <DESTINO_CONGELAMENTO> --liberacao <código>
python codigo/scripts/pair_aware_limiar.py aplicar-test \
    --execucao <RAIZ_FINAL>/pair_aware_limiar_sem_pesos_biobertpt_seed42 \
    --congelamento <DESTINO_CONGELAMENTO> --liberacao <código>
```

O primeiro passo é o único que lê o TEST: a base prevê o TEST e grava as probabilidades. O segundo aplica o t congelado. O resultado da configuração final é o do segundo passo.